In [2]:
#Importing required packages
library(dplyr)
library(tidyverse)
library(gtsummary)
library(flextable)

#Reading in dataset
df = read.csv('SR_SSU_2026.csv')
nrow(df)

[1] 1802

In [3]:
#Defining dataset
T1D_define = df %>% filter((clinical_diagnosis_v1 == 'Type 1' & num_anti ==1) | (num_anti >=2))
nrow(T1D_define)

[1] 594

In [4]:
#Correcting data formats
T1D_define$home_urine_sample_received_v1 = as.Date(T1D_define$home_urine_sample_received_v1)
T1D_define$date_of_diagnosis_v1 = as.Date(T1D_define$date_of_diagnosis_v1)
T1D_define$V2DateUrineReceive_v2 = as.Date(T1D_define$V2DateUrineReceive_v2)
T1D_define$Gender_v1 = as.factor(T1D_define$Gender_v1)
T1D_define$num_anti = as.factor(T1D_define$num_anti)
T1D_define$Smoker_v1 = as.factor(T1D_define$Smoker_v1 )
T1D_define$famhisdiab = as.factor(T1D_define$famhisdiab )
T1D_define$famhisinsdiab = as.factor(T1D_define$famhisinsdiab )
T1D_define$famhisauto = as.factor(T1D_define$famhisauto)
T1D_define$famhisnoninsdiab = as.factor(T1D_define$famhisnoninsdiab)
T1D_define$Mother_diabetes_v1 = as.factor(T1D_define$Mother_diabetes_v1)
T1D_define$Father_diabetes_v1 = as.factor(T1D_define$Father_diabetes_v1)
T1D_define$GAD_v1 = as.numeric(T1D_define$GAD_v1)
T1D_define$ZNT8_v1 = as.numeric(T1D_define$ZNT8_v1)
T1D_define$IA2_v1 = as.numeric(T1D_define$IA2_v1)

Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”


In [5]:
T1D_define = T1D_define %>%
  mutate(
    GAD_bin_v1 = factor(ifelse(GAD_bin_v1 == 1, "Yes", "No")),
    IA2_bin_v1 = factor(ifelse(IA2_bin_v1 == 1, "Yes", "No")),
    ZNT8_bin_v1 = factor(ifelse(ZNT8_bin_v1 == 1, "Yes", "No")),
    DKA = factor(ifelse(DKA == 1, "Yes", "No")),
    autoimmune = factor(ifelse(autoimmune == 1, "Yes", "No")),  
  )


In [6]:
#Extracting the duration of diabetes in years for v1 and v2
T1D_define = T1D_define %>% mutate(dur_diab_sample_yr_v1 = as.numeric(
             home_urine_sample_received_v1 - date_of_diagnosis_v1)/365.25)
T1D_define = T1D_define %>% mutate(dur_diab_sample_yr_v2 = as.numeric(
             V2DateUrineReceive_v2 - date_of_diagnosis_v1)/365.25)

#Recoding the antibody titres recorded as negatives to a fixed value below the cutoffs
T1D_define = T1D_define %>% mutate(GAD_v1 = ifelse(GAD_v1 == "negative", 10, GAD_v1))
T1D_define = T1D_define %>% mutate(IA2_v1 = ifelse(IA2_v1 == "negative", 7, IA2_v1))
T1D_define <- T1D_define %>%
  mutate(
    ZNT8_v1 = case_when(
      ZNT8_v1 == "negative" & AgeatDiagnosis < 30 ~ 64,
      ZNT8_v1 == "negative" & AgeatDiagnosis >= 30 ~ 10,
      TRUE ~ as.numeric(ZNT8_v1)
    )
  )

In [7]:
#Defining fast and slow progressors
T1D_define = T1D_define %>% mutate(c_peptide_group = ifelse(
             c_peptide_v4 <= 200,'fast progressors(≤200pmol/l)','slow progressors(>200pmol/l'))

#filtering out participants with no c_peptide measure at v4
c_peptide_clean = T1D_define %>% drop_na(c_peptide_group)

#Converting to long format, to order UCPCR repeated measurement 
ucpcr_clean = T1D_define %>%
  pivot_longer(
    cols = matches("^(UCPCR|dur_diab_sample_yr)_v\\d+$"),
    names_to = c(".value", "visit"),
    names_pattern = "(.*)_v(\\d+)"
  ) %>% arrange(Study_ID,visit)  %>% group_by(Study_ID) %>% 
    filter(any(!is.na(UCPCR))) %>% ungroup()

#Base dataset, number of participants
cat('Number of participants: ',length(unique(T1D_define$Study_ID)),'\n')
    
#Dataset for Logistic regression model, number of participants
cat('Number of participants (LMM): ',length(unique(c_peptide_clean$Study_ID)),'\n')
    
#Dataset for linear mixed effects model, number of participants
cat('Number of participants:(LOG): ',length(unique(ucpcr_clean$Study_ID)),'\n')

Number of participants:  594 
Number of participants (LMM):  408 
Number of participants:(LOG):  544 


In [15]:
#saving datasets for ucpcr and c_peptide
write.csv(c_peptide_clean,'c_peptide_clean.csv', row.names=FALSE)
write.csv(ucpcr_clean,'ucpcr_clean.csv', row.names=FALSE)

In [8]:
#Baseline characteristics of Type 1 diabetes cohort in StartRight.

tab_1 = T1D_define %>% select(AgeatDiagnosis,ethnicity,
        Gender_v1,wh_ratio_v1,c_peptide_v1,UCPCR_v1,UCPCR_v4,
        GAD_bin_v1,IA2_bin_v1, ZNT8_bin_v1, num_anti,bmi_calc_v1,
        GAD_v1, IA2_v1,ZNT8_v1,HbA1c_at_diagnosis_v1, DKA,famhisinsdiab,
        autoimmune,Smoker_v1,T1DGRS2,
        c_peptide_group) %>%
        tbl_summary(
            missing = 'no',
            type = list(
                c(GAD_bin_v1, IA2_bin_v1, ZNT8_bin_v1, DKA,
                  famhisinsdiab, autoimmune, Smoker_v1) ~ 'categorical'
                ),
            statistic = list(
                all_continuous() ~ '{mean} ({sd})',
                all_categorical() ~ '{n} ({p}%)',
                c(c_peptide_v1,UCPCR_v1,UCPCR_v4) ~'{median} ({p25},{p75})'
                ),
        digits = all_continuous() ~ 1,
        label =  list('AgeatDiagnosis' ~ 'Age at Diagnosis (years)',
               'ethnicity' ~ 'Ethnicity',
               'Gender_v1' ~ 'Gender',
               'c_peptide_group' ~ 'Blood C-peptide group',
               'GAD_bin_v1' ~ 'Glutamic acid dearboxylase positivity',
               'IA2_bin_v1' ~ 'Islet antigen 2 positivity',
               'ZNT8_bin_v1' ~ 'Zinc Transporter 8 positivity',
               'bmi_calc_v1' ~ 'Body mass index (kg/m2)',
               'HbA1c_at_diagnosis_v1' ~ 'Baseline HbA1c (mmol/mol)', 
               'famhisinsdiab' ~ 'Family history of insulin-treated diabetes',
               'DKA' ~ 'Diabetic Ketoacidosis',
               'wh_ratio_v1' ~ 'Waist hip ratio',
               'UCPCR_v1' ~ ' Baseline Urine C-peptide Creatinine ratio(nmol/mmol)',
               'UCPCR_v4' ~ 'Urine C-peptide Creatinine ratio at fourth visit (nmol/mmol)',
               'c_peptide_v1' ~ 'Baseline C-peptide (pmol/l)',
               'num_anti' ~ 'Number of Antibodies',
               'autoimmune' ~ 'History of Other auto-immune disease',
               'Smoker_v1' ~ 'Smoking Status',
               'GAD_v1' ~ 'GAD Titre (units/ml)',
               'ZNT8_v1' ~ 'ZNT8 Titre (units/ml)',
               'IA2_v1' ~ 'IA2 Titre (units/ml)'
        )) %>%
        bold_labels() %>% 
        modify_header(label = '**Characteristics**') %>%
        modify_caption('**Baseline Characteristics of Type 1 Diabetes cohort in StartRight**')


In [9]:
#Table comparing fast and slow progressors
tab2 = c_peptide_clean %>% select(AgeatDiagnosis,ethnicity,
        Gender_v1,wh_ratio_v1,c_peptide_v1,UCPCR_v1,UCPCR_v4,
        GAD_bin_v1,IA2_bin_v1, ZNT8_bin_v1, num_anti,bmi_calc_v1,
        GAD_v1, IA2_v1,ZNT8_v1,HbA1c_at_diagnosis_v1, DKA,
        autoimmune,Smoker_v1,T1DGRS2,
        c_peptide_group) %>%
        tbl_summary(
            by = c_peptide_group,
            type = list(
                c(GAD_bin_v1, IA2_bin_v1, ZNT8_bin_v1, DKA,
                  autoimmune, Smoker_v1) ~ 'categorical'
                ),
            missing = 'no',
            statistic = list(
                all_continuous() ~ '{mean} ({sd})',
                all_categorical() ~ '{n} ({p}%)',
                c(c_peptide_v1,UCPCR_v1,UCPCR_v4) ~'{median} ({p25},{p75})'
                ),
        digits = all_continuous() ~ 1,
        label =  list('AgeatDiagnosis' ~ 'Age at Diagnosis (years)',
               'ethnicity' ~ 'Ethnicity',
               'Gender_v1' ~ 'Gender',
               'bmi_calc_v1' ~ 'BMI',
               'Smoker_v1' ~ 'Smoking status',
               'wh_ratio_v1' ~ 'Waist hip ratio',
               'HbA1c_at_diagnosis_v1' ~ 'Baseline HbA1c',
               'GAD_bin_v1' ~ 'Glutamic acid dearboxylase positivity',
               'IA2_bin_v1' ~ 'Islet antigen 2 positivity',
               'ZNT8_bin_v1' ~ 'Zinc Transporter 8 positivity',
               'DKA' ~ 'Diabetic Ketoacidosis',
               'UCPCR_v1' ~ ' Baseline urine C-peptide Creatinine ratio(nmol/mmol)',
               'UCPCR_v4' ~ 'Urine C-peptide Creatinine ratio at fourth visit (nmol/mmol)',
               'c_peptide_v1' ~ 'Baseline c_peptide (pmol/l)',
               'num_anti' ~ 'Number of Antibodies',
               'autoimmune' ~ 'History of other auto-immune disease',
               'GAD_v1' ~ 'GAD Titre (units/ml)',
               'ZNT8_v1' ~ 'ZNT8 Titre (units/ml)',
               'IA2_v1' ~ 'IA2 Titre (units/ml)'
        )) %>%
        add_p(test = list(
        all_continuous() ~ 't.test',
        all_categorical() ~ 'chisq.test',
        c(c_peptide_v1,UCPCR_v1,UCPCR_v4) ~ 'wilcox.test'
        )) %>% 
        bold_labels() %>% 
        modify_header(label = '**Characteristics**')%>%
        modify_caption('**Comparison between fast and slow progressors of beta cell function decline**') %>%
        modify_table_styling(
        columns = label,
        rows = variable == "T1DGRS2",
        footnote = "T1DGRS2: Type 1 Diabetes Risk Score 2")

                      

In [10]:
tab_1 = tab_1 %>%
  modify_table_styling(
    columns = label,
    rows = variable == "wh_ratio_v1",
    footnote = "Missing data for 15 participants"
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "ethinicity",
    footnote = "Missing data for 2 participants"
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "UCPCR_v1",
    footnote = "Missing data for 108 participants"
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "UCPCR_v4",
    footnote = "Missing data for 214 participants"
  )%>%
  modify_table_styling(
    columns = label,
    rows = variable == "c_peptide_v1",
    footnote = "Missing data for 7 participants"
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "ZNT8_bin_v1",
    footnote = "Missing data for 4 participants"
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "famhisinsdiab",
    footnote = "Missing data for 1 participants"
  ) %>%
 modify_table_styling(
    columns = label,
    rows = variable == "c_peptide_group",
    footnote = "Fast progressors: Blood C-peptide ≤ 200pmol/l, Slow progressors: Blood C-peptide > 200pmol/l"
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "T1DGRS2",
    footnote = "T1DGRS2: Type 1 Diabetes Risk Score 2"
  )



In [11]:
#Characteristics of study participants
flt_1 = as_flex_table(tab_1) %>% autofit() %>%
       fit_to_width(max_width = 6.5)
save_as_docx(flt_1, path = "Table1.docx")

In [12]:
#Comparison between fast and slow progressors
flt_2 = as_flex_table(tab2) %>% fontsize(size = 10, part = "all") %>%
         padding(padding = 1, part = "all") %>% autofit()
save_as_docx(flt_2, path = "Table2.docx")

In [13]:
T1D_define = T1D_define %>% mutate(c_peptide_status = ifelse(
             c_peptide_v4 > 200, 0,1))


In [14]:
c_peptide_clean = T1D_define %>% drop_na(c_peptide_status)

In [18]:
#saving datasets for ucpcr and c_peptide
write.csv(c_peptide_clean,'c_peptide_clean.csv', row.names=FALSE)
write.csv(ucpcr_clean,'ucpcr_clean.csv', row.names=FALSE)